### Business Problem

This project aims to analyze transactional data from an e-commerce business in order to uncover insights that can help improve revenue, customer retention, and overall business performance.

E-commerce companies generate large volumes of data, but without proper analysis, it is difficult to understand customer behavior, identify high-performing products, and make data-driven decisions.

---

### Business Objectives

The main objectives of this analysis are:

- Identify the most profitable products and categories
- Understand customer purchasing behavior
- Detect patterns and trends in sales over time
- Segment customers based on their value to the business
- Provide actionable recommendations to increase revenue and customer retention

---

### Analytical Approach

To achieve these objectives, the following steps will be performed:

1. Data Cleaning and preprocessing
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Product Analysis
5. Customer Segmentation (RFM)
6. Business Insights and recommendations
7. API integration for extended analysis

---
### Expected Outcome

The final result of this project will be a set of data-driven insights and recommendations that can help the business optimize sales strategies, improve customer retention, and maximize profitability.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_excel('/kaggle/input/datasets/lakshmi25npathi/online-retail-dataset/online_retail_II.xlsx')
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

## 🧹 Data Cleaning

Before starting the analysis, the dataset was cleaned to ensure data quality and reliability.

### Steps performed:

- A copy of the original dataset was created to preserve the raw data
- Rows with negative or zero values in **Quantity** and **Price** were removed, as they do not represent valid transactions
- Missing values in the **Customer ID** column were removed, since customer-based analysis requires valid customer information
- Duplicate records were removed to avoid double counting
- The **InvoiceDate** column was converted to datetime format for time-based analysis
- A new column **Revenue** was created by multiplying Quantity and Price


In [ ]:
df_raw = df.copy()

df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

df = df.dropna(subset=['Customer ID'])

df = df.drop_duplicates()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Revenue'] = df['Quantity'] * df['Price']

print(f"Original number of records: {df_raw.shape[0]}")
print(f"Cleaned number of records: {df.shape[0]}")

df.head()

### Result:

The dataset is now cleaned and ready for further analysis, ensuring that only valid and meaningful transactions are included.

## 📊 Exploratory Data Analysis (EDA)

In this section, we explore the dataset to uncover patterns, trends, and key insights that can help understand business performance.

The goal is to analyze revenue distribution, sales trends over time, and product performance.

### Revenue by Country (Initial Overview)

To understand how revenue is distributed across different countries, we first calculate the total revenue generated by each country.

This will help us identify the top-performing markets and see whether the business depends on specific regions.

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)

print("Top 10 Countries by Revenue:")
print(country_revenue.head(10))

### Focus on International Markets

From the results above, we can observe that the United Kingdom generates significantly higher revenue compared to other countries.

To better understand the performance of other markets, we will exclude the United Kingdom and visualize the top 10 remaining countries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top10_df = country_revenue.iloc[1:11].reset_index()
top10_df.columns = ["Country", "Revenue"]

plt.figure(figsize=(12, 6))
sns.set(style="whitegrid")

ax = sns.barplot(
    data=top10_df, 
    x="Revenue", 
    y="Country", 
    hue="Country", 
    palette="viridis", 
    legend=False
)
for i in ax.containers:
    ax.bar_label(i, fmt='%.0f', padding=5)

plt.xlim(0, top10_df['Revenue'].max() * 1.25)
plt.title('Top 10 International Markets by Revenue (Excluding UK)', fontsize=15, pad=20)
plt.xlabel('Total Revenue', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.show()

### Insight:
The United Kingdom dominates overall revenue, so it was excluded from the visualization to better highlight the performance of other international markets.

Among the remaining countries, a small group contributes the majority of revenue, indicating potential key markets for expansion.

### Revenue Trend Over Time

After analyzing the geographical distribution of revenue, it is important to examine how revenue evolves over time.

This analysis helps identify trends, seasonality patterns, and periods of growth or decline in the business performance.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Month_Year'] = df['InvoiceDate'].dt.to_period('M')

monthly_sales = df.groupby('Month_Year')['Revenue'].sum().reset_index()

monthly_sales['Month_Year'] = monthly_sales['Month_Year'].astype(str)

plt.figure(figsize=(14, 6))

plt.plot(monthly_sales['Month_Year'], monthly_sales['Revenue'], marker='o', linestyle='-', color='royalblue', linewidth=2)
plt.title('Monthly Revenue Trend (2009 - 2011)', fontsize=16, pad=20)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Revenue', fontsize=12)

plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Insight:
The revenue shows noticeable fluctuations over time, indicating periods of higher and lower sales activity.

There may be seasonal patterns, with certain months generating higher revenue, which could be linked to holidays or promotional periods.

Understanding these trends can help the business better plan marketing strategies and inventory management.

### Best-Selling Products

To understand product performance, we analyze which products are sold in the highest quantities.

This helps identify popular items and products that drive sales volume.

In [ ]:
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=top_products, 
    x="Quantity", 
    y="Description", 
    hue="Description",
    palette="magma", 
    legend=False
)

for i in ax.containers:
    ax.bar_label(i, fmt='%.0f', padding=5)

plt.title('Top 10 Best Selling Products (by Quantity)', fontsize=16, pad=20)
plt.xlabel('Total Quantity Sold', fontsize=12)
plt.ylabel('Product Description', fontsize=12)
plt.xlim(0, top_products['Quantity'].max() * 1.2)
plt.show()

### Insight:
Sales volume is concentrated in a small number of products, suggesting that the business relies on a few highly popular items.

These products are likely essential for maintaining consistent sales and should be carefully managed in terms of stock and availability.

## ⚙️ Feature Engineering

In this step, new features are created from the existing data to enhance the analysis.

These features will help better understand customer behavior, purchasing patterns, and overall business performance.

In [ ]:
orders_per_customer = df.groupby('Customer ID')['Invoice'].nunique().reset_index()
orders_per_customer.columns = ['CustomerID', 'TotalOrders']
orders_per_customer

In [ ]:
revenue_per_customer = df.groupby('Customer ID')['Revenue'].sum().reset_index()
revenue_per_customer.columns = ['CustomerID', 'TotalRevenue']
revenue_per_customer

In [ ]:
customer_df = pd.merge(orders_per_customer, revenue_per_customer, on = 'CustomerID')
customer_df

In [ ]:
customer_df['AvgOrderValue'] = customer_df['TotalRevenue'] / customer_df['TotalOrders']
customer_df

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days = 1)
recency_df = df.groupby('Customer ID')['InvoiceDate'].max().reset_index()
recency_df['Recency'] = (snapshot_date - recency_df['InvoiceDate']).dt.days

customer_df = pd.merge(customer_df, recency_df[['Customer ID', 'Recency']],
                      left_on = 'CustomerID', right_on = 'Customer ID').drop('Customer ID', axis=1)

customer_df.head()

### Feature Engineering Summary

New customer-level features were created to better understand purchasing behavior:

- Total number of orders per customer  
- Total revenue generated by each customer  
- Average order value  
- Recency (number of days since the customer's last purchase)

These features provide a more complete view of customer behavior by combining purchase frequency, spending patterns, and recency of activity.

They will be used in the next steps to identify high-value customers, detect potentially inactive customers, and support data-driven business decisions.

## 🛍️ Product Analysis

In this section, we analyze product performance to identify which items contribute most to the business revenue.

Understanding top-performing products helps the business optimize inventory, pricing strategies, and marketing efforts.

### Comparing Product Performance

To gain deeper insights into product performance, we compare the top products based on revenue and quantity sold.

This helps identify whether the most sold products are also the most profitable, or if there are differences between sales volume and revenue generation.

In [ ]:
top10_revenue = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()
top10_quantity = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).reset_index()



fig, ax = plt.subplots(1, 2, figsize = (18, 8))

sns.barplot(
    data = top10_revenue, 
    x = 'Revenue', 
    y = 'Description', 
    hue = 'Description',
    palette = 'viridis', 
    ax = ax[0], 
    legend = False
)
ax[0].set_title('Top 10 Products by Total Revenue', fontsize = 15, pad=15)
ax[0].set_xlabel('Revenue')
ax[0].set_ylabel('Description')

sns.barplot(
    data = top10_quantity, 
    x = 'Quantity', 
    y = 'Description', 
    hue = 'Description',
    palette = 'magma', 
    ax = ax[1], 
    legend = False
)
ax[1].set_title('Top 10 Products by Total Quantity', fontsize = 15, pad=15)
ax[1].set_xlabel('Quantity Sold')
ax[1].set_ylabel('Description')

plt.tight_layout()
plt.show()

### Insight:

There is a noticeable difference between the most sold products and the most profitable ones.

Some products generate high revenue despite lower sales volume, likely due to higher prices.  
Others are sold frequently but contribute less to total revenue.

This suggests that both pricing and sales volume play an important role in overall business performance.

### Revenue and Market Distribution by Country

To better understand international market performance, we analyze both revenue and transaction distribution across countries (excluding the United Kingdom).

This helps identify key markets and compare their contribution to overall business activity.

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).reset_index()
country_revenue_no_uk = country_revenue[country_revenue['Country'] != 'United Kingdom'].head(10)

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.barplot(
    data=country_revenue_no_uk, 
    x='Revenue', 
    y='Country', 
    hue='Country', 
    palette='viridis', 
    legend=False)
plt.title('Top Countries by Revenue (Excluding UK)', fontsize=14)
plt.xlabel('Total Revenue')

plt.subplot(1, 2, 2)
country_counts = df[df['Country'] != 'United Kingdom']['Country'].value_counts().head(5)
plt.pie(
    country_counts, 
    labels=country_counts.index, 
    autopct='%1.1f%%', 
    colors=sns.color_palette('pastel'))
plt.title('Market Share by Transactions (Top 5 Countries)', fontsize=14)

plt.tight_layout()
plt.show()

### Insight:
Revenue and transaction distribution show that a small number of countries dominate international sales.

Some countries generate higher revenue, while others have a higher share of transactions, indicating differences in purchasing behavior across markets.

This suggests opportunities for targeted strategies in specific regions.

### Top Products by Country

To better understand customer preferences across different markets, we analyze the top-selling products in selected countries.

This helps identify differences in demand and highlights how product popularity varies by region.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

top_eire = df[df['Country'] == 'EIRE'].groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
sns.barplot(
    x=top_eire.values, 
    y=top_eire.index, 
    ax=ax[0], 
    palette='magma', 
    hue=top_eire.index, 
    legend=False)
ax[0].set_title('Top 5 in EIRE')

top_germany = df[df['Country'] == 'Germany'].groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
sns.barplot(
    x=top_germany.values, 
    y=top_germany.index, 
    ax=ax[1], 
    palette='magma', 
    hue=top_germany.index, 
    legend=False)
ax[1].set_title('Top 5 in Germany')

top_france = df[df['Country'] == 'France'].groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
sns.barplot(
    x=top_france.values, 
    y=top_france.index, 
    ax=ax[2], 
    palette='magma', 
    hue=top_france.index, 
    legend=False)
ax[2].set_title('Top 5 in France')

plt.tight_layout()
plt.show()

### Insight:
Product preferences vary between countries, as different items appear among the top-selling products in each market.

This indicates that customer behavior is not uniform across regions and suggests the need for localized product strategies.

## 👥 Customer Segmentation

In this section, we analyze customer behavior to better understand how customers interact with the business.

The analysis focuses on:
- How recently customers have made a purchase (Recency)  
- How often they purchase (Frequency)  
- How much they spend (Monetary value)  

This helps identify valuable customers, loyal customers, and those who may need re-engagement.

### Top Customers by Revenue

To identify the most valuable customers, we analyze those who generate the highest total revenue.

In [ ]:
top_customers = customer_df.sort_values(by = 'TotalRevenue', ascending = False).head(10)
top_customers

### Insight:
A small group of customers generates a significant portion of total revenue.

These customers are highly valuable to the business and should be prioritized in retention and loyalty strategies.

### Customer Activity (Recency)

To understand customer engagement, we analyze how recently customers have made a purchase.

In [ ]:
customer_df['Recency'].describe()

### Insight:
Customers with lower Recency values are more active and engaged.

Customers with higher Recency values may be at risk of not returning and could require targeted marketing efforts.

In [ ]:
plt.figure(figsize = (10, 6))
sns.scatterplot(
    data = customer_df,
    x = 'TotalOrders',
    y = 'TotalRevenue',
    alpha = 0.6,
    color='dodgerblue'
)
plt.title('Customer Segmentation: Frequency vs Monetary')
plt.xlabel('Number of Orders (Frequency)') 
plt.ylabel('Total Revenue (Monetary)')

plt.xlim(0, customer_df['TotalOrders'].quantile(0.99))
plt.ylim(0, customer_df['TotalRevenue'].quantile(0.99))
plt.grid(True, linestyle = '--', alpha = 0.5)

plt.show()

### Insight:
Most customers generate relatively low revenue and place a small number of orders.

A smaller group of customers shows higher frequency and spending, indicating more valuable and engaged customers.

In [ ]:
def simple_segmenter(days):
    if days <= 30:
        return 'Active'
    elif days <= 90:
        return 'At Risk'
    else:
        return 'Churned'

customer_df['Status'] = customer_df['Recency'].apply(simple_segmenter)
print(customer_df['Status'].value_counts())

In [ ]:
status_counts = customer_df['Status'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(status_counts, 
        labels=status_counts.index,
        autopct='%1.1f%%',              
        colors=sns.color_palette('pastel')
       ) 
plt.title('Customer Base by Retention Status', fontsize=14)
plt.show()

### Insight:
Customers can be divided into active, at-risk, and churned groups based on their recent activity.

A significant number of customers may fall into the at-risk or churned categories, indicating potential opportunities for re-engagement strategies.

## 📌 Conclusion & Business Insights

Based on the analysis, several key insights and actionable recommendations can be made to improve business performance.

### Key Business Insights & Recommendations

- **Focus on top-performing products**  
A small number of products generate a large share of revenue. Increasing their visibility and ensuring availability can boost sales.

- **Strengthen international markets**  
Revenue is highly concentrated in the United Kingdom. Expanding marketing efforts in other countries could increase overall growth.

- **Retain high-value customers**  
A small group of customers contributes significantly to total revenue. Loyalty programs and personalized offers can help retain them.

- **Re-engage inactive customers**  
Customers classified as “At Risk” or “Churned” should be targeted with campaigns, discounts, or reminders to return.

- **Adapt strategy by region**  
Product preferences differ across countries, suggesting the need for localized marketing and product selection.

## 🔌 API Integration

To enhance the analysis, we demonstrate how external data can be integrated using an API.

This allows for real-time data updates and automation of insights.

In [ ]:
import requests

url = "https://api.exchangerate-api.com/v4/latest/GBP"
response = requests.get(url)
data = response.json()

print("Exchange rates (GBP):")
print(data['rates'])

### Insight:
Using APIs allows businesses to access real-time external data, such as exchange rates, which can support better decision-making and dynamic pricing strategies.